## The unbounded table model

Most data work is **batch** — read a file, transform, write a result. The world produces data continuously: card transactions every millisecond, fraud alerts, ATM withdrawals, sensor readings. Waiting for tomorrow's batch means decisions are always stale. **Structured Streaming** brings the DataFrame API to continuous, unbounded data.

The mental model: treat the input stream as an **infinite table** that grows row by row. You write a query against that table — the same `select`, `filter`, `groupBy`, `join` you already know — and Spark runs it continuously as new data arrives in **micro-batches**.

```text
Input Stream  ────────►  growing infinite table
                              │
                              ▼
                       Your query (filter, window, agg, join)
                              │
                              ▼
                          Output table / Sink
```

Key vocabulary:

| Term | Meaning |
|---|---|
| **Source** | Where data comes from: rate, files, Kafka, socket |
| **Micro-batch** | One chunk of new data processed in a single Spark job |
| **Trigger** | How often Spark checks for new data |
| **Output mode** | Which rows are written each trigger (append / update / complete) |
| **Sink** | Where results go: memory, console, file, Delta, Kafka, foreachBatch |
| **Checkpoint** | State + offsets persisted to disk so the query can restart |
| **Watermark** | How long to wait for late events before closing a window |

The capstone project at `project/06-stream-fraud-detect.ipynb` builds this end to end on bank transactions; this notebook teaches the primitives.

## Setup

Streaming demos need a scratch area for source files, sinks, and checkpoints. We use `data/_streaming/`.

In [ ]:
import os, json, time, shutil, pprint
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import (
    col, lit, when, count, sum as _sum, window,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType, TimestampType,
)

builder = (
    SparkSession.builder
    .appName("StructuredStreaming")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")     # smaller for local demos
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "data"
SCRATCH  = os.path.abspath(os.path.join(DATA_DIR, "_streaming"))
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

def scratch(*parts):
    p = os.path.join(SCRATCH, *parts)
    os.makedirs(p, exist_ok=True)
    return p

def drop_txns(dirpath, batch_id, rows):
    """Write a JSON file of transactions to a watched directory."""
    path = os.path.join(dirpath, f"batch_{batch_id:04d}.json")
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    return path

print("scratch:", SCRATCH)

## Your first streaming query — rate source → memory sink

The **rate source** generates rows at a fixed rate with two columns: `timestamp` and `value`. The **memory sink** collects results into an in-memory temp view you can query with SQL. Together, they're the simplest end-to-end streaming demo — no external system needed.

`writeStream` chains:

- `.format("memory")` — sink type
- `.queryName(name)` — register as `<name>` for SQL
- `.outputMode("append")` — emit new rows only
- `.start()` — actually starts the query and returns a `StreamingQuery` handle

We call `.awaitTermination(5)` to let the query run for 5 seconds, then `.stop()`.

In [ ]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

q = (
    rate_df
    .filter(col("value") % 2 == 0)
    .writeStream
    .format("memory")
    .queryName("even_values")
    .outputMode("append")
    .start()
)

q.awaitTermination(5)
q.stop()

spark.sql("SELECT COUNT(*) AS rows, MIN(value) AS lo, MAX(value) AS hi FROM even_values").show()

## Sources — rate, file, Kafka (reference)

A **source** is the entry point of a streaming pipeline. Spark tracks source progress (offsets / file lists) in the checkpoint so each micro-batch is processed exactly once on restart.

| Source | Format string | Fault-tolerant | Best used for |
|---|---|---|---|
| **Rate** | `"rate"` | Yes | Testing, benchmarking, demos |
| **Rate per microbatch** | `"rate-micro-batch"` | Yes | Deterministic testing |
| **File** | `"json"` / `"csv"` / `"parquet"` / `"orc"` / `"delta"` | Yes | File-drop pipelines, ETL |
| **Kafka** | `"kafka"` | Yes | Production event streaming |
| **Socket** | `"socket"` | **No** | Interactive demos only |

**File source key options:** `maxFilesPerTrigger` (rate-limit), `latestFirst` (catch-up), `cleanSource = "delete"` or `"archive"`. Schema must be provided **explicitly** — Spark cannot infer for streaming file sources.

**Kafka source — reference snippet (not runnable here):**

```python
kafka_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe",               "card_transactions")
    .option("startingOffsets",         "latest")        # or "earliest"
    .option("maxOffsetsPerTrigger",    10000)
    .load()
)
# Kafka rows: key/value (binary) + topic/partition/offset/timestamp
parsed = kafka_stream.select(
    from_json(col("value").cast("string"), txn_schema).alias("d")
).select("d.*")
```

The capstone uses the file-source pattern. We'll demo it next.

In [ ]:
# A landing directory the stream will watch
LANDING   = scratch("landing")
CKPT_FILE = scratch("ckpts", "file_demo")

txn_schema = StructType([
    StructField("transaction_id",    StringType()),
    StructField("customer_id",       StringType()),
    StructField("merchant_category", StringType()),
    StructField("amount",            DecimalType(18, 2)),
    StructField("status",            StringType()),
    StructField("transaction_at",    TimestampType()),
])

# Drop a few files BEFORE starting the query
drop_txns(LANDING, 1, [
    {"transaction_id": "T0001", "customer_id": "CUST001", "merchant_category": "Groceries",
     "amount": 1200.00,  "status": "APPROVED", "transaction_at": "2026-05-08T09:00:00"},
    {"transaction_id": "T0002", "customer_id": "CUST002", "merchant_category": "Travel",
     "amount": 18500.00, "status": "APPROVED", "transaction_at": "2026-05-08T09:00:05"},
])
drop_txns(LANDING, 2, [
    {"transaction_id": "T0003", "customer_id": "CUST001", "merchant_category": "Food",
     "amount": 450.00,   "status": "APPROVED", "transaction_at": "2026-05-08T09:00:10"},
])

# trigger(availableNow=True) processes all files and stops
txn_stream = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 10)
    .json(LANDING)
)

q = (
    txn_stream.writeStream
    .format("memory").queryName("file_txns")
    .outputMode("append")
    .option("checkpointLocation", CKPT_FILE)
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

spark.sql("SELECT * FROM file_txns ORDER BY transaction_id").show(truncate=False)

## Sinks — memory, console, file, Delta, Kafka, foreachBatch

A **sink** is where each micro-batch's output goes. Sinks differ in supported output modes and fault-tolerance guarantees.

| Sink | Format | Output modes | Fault-tolerant | Use for |
|---|---|---|---|---|
| **memory** | `"memory"` | append, complete | No | Dev / testing |
| **console** | `"console"` | append, update, complete | No | Dev / debugging |
| **file** | `"parquet"` etc. | append only | Yes | Immutable file pipelines |
| **Delta** | `"delta"` | append, update, complete | **Yes** | Production, mutable tables |
| **Kafka** | `"kafka"` | append, update, complete | Yes | Event forwarding |
| **foreach** | `.foreach(ForeachWriter)` | all | Depends | Row-level custom logic |
| **foreachBatch** | `.foreachBatch(func)` | all | Depends | Batch-level custom logic, multi-table writes |

The production default is **Delta** — supports all three output modes, gives exactly-once via transactional commits, and avoids the small-file problem the file sink suffers from.

**`foreachBatch`** is the escape hatch for "this micro-batch goes into three different tables" or "I need to call a non-Spark API per batch." It receives the batch as a regular DataFrame plus a `batch_id`.

In [ ]:
DELTA_OUT  = scratch("delta_sink")
CKPT_DELTA = scratch("ckpts", "delta_demo")

q = (
    spark.readStream.schema(txn_schema).json(LANDING)
    .filter(col("status") == "APPROVED")
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CKPT_DELTA)
    .trigger(availableNow=True)
    .start(DELTA_OUT)
)
q.awaitTermination()

print("rows in delta sink:", spark.read.format("delta").load(DELTA_OUT).count())

In [ ]:
# foreachBatch — route each micro-batch into multiple destinations
CKPT_FB = scratch("ckpts", "foreachbatch_demo")

def route_by_status(batch_df, batch_id):
    approved = batch_df.filter(col("status") == "APPROVED")
    declined = batch_df.filter(col("status") != "APPROVED")
    n_app = approved.count()
    n_dec = declined.count()
    if n_app:
        approved.write.mode("append").format("delta").save(scratch("approved_out"))
    if n_dec:
        declined.write.mode("append").format("delta").save(scratch("declined_out"))
    print(f"batch {batch_id}: approved={n_app}, declined={n_dec}")

q = (
    spark.readStream.schema(txn_schema).json(LANDING)
    .writeStream
    .foreachBatch(route_by_status)
    .option("checkpointLocation", CKPT_FB)
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

## Output modes — append, update, complete

The output mode controls **which rows** Spark writes to the sink in each micro-batch.

| Mode | Rows emitted per trigger | Use when |
|---|---|---|
| **append** (default) | Only new rows added since the last trigger | Stateless ops, or windowed aggregations with a watermark (closed windows are emitted once) |
| **update** | Only rows that changed since the last trigger | Aggregations where you want incremental updates |
| **complete** | The entire result table, rewritten each trigger | Small aggregations written to memory or an overwrite sink |

Rules:

- `append` works for stateless operations always; for windowed aggregations it requires a **watermark** so Spark knows when a window is "done."
- `complete` requires an aggregation; state grows forever (Spark must hold the full result).
- `update` requires an aggregation; not all sinks support it (file sinks reject it; Delta accepts it).

## Triggers — controlling micro-batch cadence

A **trigger** defines how often Spark checks for new data and runs a micro-batch.

| Trigger | API | Behavior |
|---|---|---|
| **Default** (unspecified) | — | Run next batch as soon as previous one finishes |
| **Fixed interval** | `.trigger(processingTime="30 seconds")` | Run every 30s; skip if previous batch is still running |
| **Once** | `.trigger(once=True)` | Process all available data in **one** batch, then stop |
| **Available now** | `.trigger(availableNow=True)` | Process all available data in **multiple** optimally-sized batches, then stop |
| **Continuous** | `.trigger(continuous="1 second")` | Experimental ~1ms-latency mode |

`once` and `availableNow` turn a streaming query into a self-terminating batch job — you keep streaming semantics (exactly-once + offset tracking + idempotent restart), but the job exits once caught up. Most cells in this notebook use `availableNow=True` for this reason.

In [ ]:
# Drop more files; process all + stop with availableNow=True
drop_txns(LANDING, 3, [
    {"transaction_id": "T0010", "customer_id": "CUST003", "merchant_category": "Shopping",
     "amount": 6700.00, "status": "APPROVED", "transaction_at": "2026-05-08T09:01:00"},
    {"transaction_id": "T0011", "customer_id": "CUST002", "merchant_category": "Fuel",
     "amount": 2800.00, "status": "DECLINED", "transaction_at": "2026-05-08T09:01:05"},
])

q = (
    spark.readStream.schema(txn_schema).json(LANDING)
    .writeStream.format("memory").queryName("available_now_demo")
    .outputMode("append")
    .option("checkpointLocation", scratch("ckpts", "availnow"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()
print("processed rows:", spark.sql("SELECT COUNT(*) FROM available_now_demo").collect()[0][0])

## Checkpointing and fault tolerance

Checkpointing is what makes Structured Streaming **exactly-once** and **restartable**. Without it, a crashed query restarts from scratch and may double-process data.

What's stored in the checkpoint directory:

```text
<checkpointLocation>/
├── commits/        ← which batches are committed to the sink
├── offsets/        ← source positions (file index, Kafka offset) per batch
├── metadata        ← query metadata
└── state/          ← state store contents (for stateful queries)
```

**Always set `checkpointLocation`** on any production query. On restart, Spark resumes exactly where it left off — no data is reprocessed or skipped.

End-to-end exactly-once requires both **source** and **sink** to support it:

| Source | Sink | Guarantee |
|---|---|---|
| Kafka | Delta Lake | Exactly-once |
| Kafka | Kafka (idempotent producer) | Exactly-once |
| File | Parquet / Delta | Exactly-once |
| Rate | Memory / Console | At-least-once (sinks not durable) |
| Socket | Any | At-most-once (source not replayable) |

## Monitoring streaming queries

The `StreamingQuery` handle returned by `.start()` exposes everything you need to observe a running stream.

| Property | Returns |
|---|---|
| `query.isActive` | `True` if still running |
| `query.id` / `query.runId` | Stable / per-run UUIDs |
| `query.status` | Dict: `{message, isDataAvailable, isTriggerActive}` |
| `query.lastProgress` | Dict: timing, input rates, state rows, watermark |
| `query.recentProgress` | Last 100 progress dicts |
| `query.awaitTermination(timeout)` | Block until stopped or timeout |
| `query.stop()` | Stop gracefully |
| `query.exception()` | Returns exception if failed |
| `spark.streams.active` | All active queries in the session |

In [ ]:
q = (
    spark.readStream.format("rate").option("rowsPerSecond", 10).load()
    .writeStream.format("memory").queryName("monitor_demo")
    .start()
)
time.sleep(3)
prog = q.lastProgress or {}
print("name             :", prog.get("name"))
print("batchId          :", prog.get("batchId"))
print("inputRowsPerSec  :", prog.get("inputRowsPerSecond"))
print("numInputRows     :", prog.get("numInputRows"))
print("durationMs       :", prog.get("durationMs"))
q.stop()

## Time and windows — tumbling, sliding, session

Two clocks in any streaming pipeline:

- **Processing time** — when Spark sees the row.
- **Event time** — when the event actually happened in the source system. The honest clock for analytics; what `withWatermark` operates on.

Three window types over event time:

| Type | API | Semantics |
|---|---|---|
| **Tumbling** | `window(ts, "5 minutes")` | Fixed, non-overlapping windows |
| **Sliding** | `window(ts, "5 minutes", "1 minute")` | Fixed-size, overlapping windows that advance every slide |
| **Session** | `session_window(ts, "10 minutes")` | Activity-driven; opens on first event, closes after `gap_duration` of silence |

Use **session windows** for "did the customer have a burst of activity?" patterns — fraud detection, user session analytics, device heartbeats. Available since Spark 3.2.

In [ ]:
# Tumbling 1-minute windows over event time on the file stream
windowed = (
    spark.readStream.schema(txn_schema).json(LANDING)
    .filter(col("status") == "APPROVED")
    .groupBy(
        window(col("transaction_at"), "1 minute"),
        col("merchant_category"),
    )
    .agg(count("*").alias("n_txns"), _sum("amount").alias("total_amount"))
)

q = (
    windowed.writeStream
    .format("memory").queryName("windowed_demo")
    .outputMode("complete")            # complete works without watermark
    .option("checkpointLocation", scratch("ckpts", "windowed"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

spark.sql("""
    SELECT window.start AS win_start, merchant_category, n_txns, total_amount
    FROM   windowed_demo
    ORDER BY win_start, merchant_category
""").show(truncate=False)

## Watermarking — how it works

Late events break windowed aggregations: a window that "closed" 5 minutes ago can't quietly accept a row that drifts in 3 minutes late. Watermarking is the mechanism for handling this.

```text
watermark = max(event_time seen so far) − delay
```

- Events whose event time is **before the watermark** are **dropped**.
- Windows whose end time is **before the watermark** are **closed** — state freed, result emitted.

Set with `.withWatermark(col, "10 minutes")` *before* the `groupBy`.

**Output mode × watermark:**

| Mode | Watermark | When result emitted | When state freed |
|---|---|---|---|
| **complete** | Optional | Every trigger (full table) | Never |
| **update** | Recommended | Every trigger if changed | After watermark passes window end |
| **append** | **Required** for windowed agg | Only after window closes | After emission |

**Choosing the duration** is a business SLA decision: longer = more late data accepted but more memory and later finalization; shorter = faster results but lost stragglers. Practical guideline: measure the p99 event-time spread per micro-batch in your pipeline; that's your floor.

In [ ]:
# Watermarked windowed agg — update mode emits incrementally
windowed_w = (
    spark.readStream.schema(txn_schema).json(LANDING)
    .filter(col("status") == "APPROVED")
    .withWatermark("transaction_at", "10 minutes")
    .groupBy(
        window(col("transaction_at"), "1 minute"),
        col("merchant_category"),
    )
    .count()
)

q = (
    windowed_w.writeStream
    .format("memory").queryName("watermarked_demo")
    .outputMode("update")
    .option("checkpointLocation", scratch("ckpts", "watermarked"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

spark.sql("SELECT * FROM watermarked_demo ORDER BY window").show(truncate=False)

## Joins — stream-static and stream-stream

**Stream-static join** — enrich each streaming row with a static dimension table. The static side is broadcast; no watermark needed; supports all output modes. The static DataFrame is re-read every micro-batch, so changes to the static data eventually propagate.

**Stream-stream join** — both sides live; both must have **watermarks** so unmatched rows can be evicted from state. Typically also a **time-range condition** (`A.ts BETWEEN B.ts - 5 min AND B.ts + 5 min`) so Spark knows when matches become impossible.

| Join type | Stream-stream supported | Notes |
|---|---|---|
| inner | Yes | Watermarks + range condition required |
| left outer | Yes | Right unmatched emitted after watermark advances |
| right outer | Yes | Left unmatched emitted after watermark advances |
| full outer | **Not supported** | — |

In [ ]:
# Stream-static — read once, broadcast-joined to the stream each batch
static_customers = spark.read.option("header", "true").csv(f"{DATA_DIR}/customers")

enriched = (
    spark.readStream.schema(txn_schema).json(LANDING)
    .join(static_customers, "customer_id", "left")
    .select("transaction_id", "customer_id", "full_name", "city", "amount", "status")
)

q = (
    enriched.writeStream
    .format("memory").queryName("enriched_demo")
    .outputMode("append")
    .option("checkpointLocation", scratch("ckpts", "enriched"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

spark.sql("SELECT * FROM enriched_demo ORDER BY transaction_id").show(truncate=False)

## Arbitrary stateful — `applyInPandasWithState`, timeouts, dedup

Built-in `groupBy + agg` and `window` cover most cases. When you need **arbitrary per-key state** — a state machine, a custom expiry policy, a counter that doesn't map cleanly to an aggregate — reach for `applyInPandasWithState` (Spark 3.4+).

The function receives the **key**, a **pandas iterator** of new rows for this key in this micro-batch, and the **current state**. It returns updated state and zero-or-more output rows.

**Pattern (snippet — runnable demo below uses simpler dedup):**

```python
def detect_fraud_burst(key, rows_iter, state):
    n = state.get[0] if state.exists else 0
    for batch in rows_iter:
        n += len(batch)
    state.update((n,))
    if n >= 3:
        yield pd.DataFrame([{"customer_id": key[0], "burst": n}])
```

**Timeouts** — without them, state for inactive keys lives forever:

| Timeout | Clock | Fires when |
|---|---|---|
| `ProcessingTimeTimeout` | wall clock | N ms after `setTimeoutDuration` |
| `EventTimeTimeout` | watermark | watermark passes the timestamp set by `setTimeoutTimestamp` |

**Stateful deduplication** — Kafka and most sources are at-least-once. `dropDuplicates` with a watermark drops re-delivered messages within the watermark window without growing state forever:

```python
.withWatermark("transaction_at", "10 minutes")
.dropDuplicates(["transaction_id", "transaction_at"])
```

The capstone in `project/06-stream-fraud-detect.ipynb` uses the same pattern — watermark plus a velocity rule per customer.

In [ ]:
# Drop the same transaction twice — stateful dedup keeps only one
DEDUP_LANDING = scratch("dedup_landing")
drop_txns(DEDUP_LANDING, 1, [
    {"transaction_id": "DUP01", "customer_id": "CUST001", "merchant_category": "Food",
     "amount": 250.00,  "status": "APPROVED", "transaction_at": "2026-05-08T10:00:00"},
    {"transaction_id": "DUP01", "customer_id": "CUST001", "merchant_category": "Food",
     "amount": 250.00,  "status": "APPROVED", "transaction_at": "2026-05-08T10:00:00"},
    {"transaction_id": "DUP02", "customer_id": "CUST002", "merchant_category": "Travel",
     "amount": 4500.00, "status": "APPROVED", "transaction_at": "2026-05-08T10:01:00"},
])

deduped = (
    spark.readStream.schema(txn_schema).json(DEDUP_LANDING)
    .withWatermark("transaction_at", "10 minutes")
    .dropDuplicates(["transaction_id", "transaction_at"])
)

q = (
    deduped.writeStream
    .format("memory").queryName("deduped_demo")
    .outputMode("append")
    .option("checkpointLocation", scratch("ckpts", "deduped"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

spark.sql("SELECT * FROM deduped_demo ORDER BY transaction_id").show(truncate=False)

## State store backends + monitoring state

Spark stores stateful streaming state in a **state store**. Two backends are available:

| Backend | Config | Characteristics |
|---|---|---|
| **HDFS-backed** (default) | `HDFSBackedStateStoreProvider` | State in JVM heap; checkpointed each batch. Simple. Heap-bound. |
| **RocksDB** | `RocksDBStateStoreProvider` | State in on-disk RocksDB; checkpointed each batch. Handles GBs of state with low heap pressure. |

Switch to RocksDB when state exceeds a few GB per executor or when stateful stages show GC pressure / OOMs.

```python
.config(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider",
)
```

**Monitoring state** — every stateful query reports `stateOperators` in `lastProgress`:

| Field | Meaning |
|---|---|
| `numRowsTotal` | Rows currently in the state store |
| `numRowsUpdated` | Rows added/updated this micro-batch |
| `numRowsRemoved` | Rows evicted (timeout / watermark) this micro-batch |
| `memoryUsedBytes` | State memory in bytes |
| `numShufflePartitions` | State partition count |
| `operatorName` | e.g. `stateStoreSave`, `symmetricHashJoin`, `dedupe` |

In [ ]:
print("provider class     :", spark.conf.get("spark.sql.streaming.stateStore.providerClass", "(default)"))
print("shuffle partitions :", spark.conf.get("spark.sql.shuffle.partitions"))

# Run a quick stateful query so lastProgress.stateOperators is populated
quick = (
    spark.readStream.format("rate").option("rowsPerSecond", 50).load()
    .withWatermark("timestamp", "10 seconds")
    .groupBy(window(col("timestamp"), "5 seconds"))
    .count()
)

q = (
    quick.writeStream.format("memory").queryName("state_metrics_demo")
    .outputMode("update")
    .option("checkpointLocation", scratch("ckpts", "state_metrics"))
    .start()
)
time.sleep(4)
prog = q.lastProgress or {}
q.stop()

print("\n=== stateOperators ===")
pprint.pp(prog.get("stateOperators", []))

In [ ]:
# Stop any remaining queries; uncomment the rmtree to wipe the scratch dir.
for q in spark.streams.active:
    q.stop()

# shutil.rmtree(SCRATCH, ignore_errors=True)
spark.stop()